# Getting started

Welcome to the Getting Started guide of `l2co`! This document will help you set up your optimization tasks, optimizers and get familiar with the basics.

## Table of Contents

- [Task class](#task-class)
- [Optimizer class](#optimizer-class)
- [Budget function](#budget-function)
- [Samplers](#samplers)
- [Running optimization](#running-optimization)
- [Analyzing results](#analyzing-results)

## Task class

The [`Task`] class is one of the core components of `l2co`, representing the optimization problem you want to solve. It encapsulates the following elements:

- **objective Function** (`loss_fn`): The function you want to minimize. Currently, only single-objective and minimization tasks are supported. The function should take a single PyTree (`eqx.Module` or nested structures of arrays) as the first positional input and return a scalar loss value.
- **model** (`model`): The model or parameters that you want to optimize. This is typically an instance of `eqx.Module` or a nested structure of arrays.
- **global_min** (`global_min`): The known global minimum value of the objective function, if available. This is used for computing performance metrics such as the Expected Running Time (ERT). Set to `None` if unknown.
- **data** (`dataset`): Optional data that may be required by the objective function. This is presented as a dictionary with three keys:
    - **dataset_path**: path to the dataset. The dataset will be lazily-loaded when the `loaded_dataset` property is accessed. The data needs to be in a dictionary format with each key represting the name of a data field and the value being jax array with the leading dimension representing the number of samples.
    - **batch_size**: size of the batches to be used during training. If `None`, the entire dataset will be used as a single batch.
    - **seed**: random seed for shuffling the dataset before creating batches.
- **has_aux**: A boolean flag indicating whether the objective function returns auxiliary information along with the loss value. If `True`, the objective function should return a tuple where the first element is the loss and the second element is a dictionary of auxiliary information.
- **tag**: An optional dictionary with string keys that can be used to identify the task.
- **pass_rng**: A boolean flag indicating whether to pass a random number generator (RNG) to the objective function. This can be used to mimic aleatoric uncertainty in loss function.


In [ ]:
from l2co_tasks import Task

The sibling `l2co_tasks` package contains several built-in tasks for optimization problems. You can also create custom tasks by instantiating the `Task` class and providing your own objective functions and dataset:

In [ ]:
from l2co_tasks import create_bbob_task


rastrigin_function = create_bbob_task(fn_name='rastrigin', seed=42, dimensionality=10)
sphere_function = create_bbob_task(fn_name='sphere', seed=42, dimensionality=10)

rastrigin_function

The following built-in tasks are available in `l2co_tasks`:

- benchmark function tasks using the BBOB suite from `bbob_jax`
- cifar10 image classification task
- classification of random gaussians
- mnist1d image classification task
- PDE-based tasks (reaction, convection, diffusion)
- fitting of quadratic functions
- training a RNN for spirals classification

The `tag` parameter is useful for training the `l2co` model, as it is used as input to the classification model. Tasks with similar tags are considered to have similar loss landscapes, and are threated as different realizations of the same task.

The `Task` object can be saved to a JSON file (`Task.save`) and loaded again (`Task.load`) for later use.

## Optimizer class

The [`OptimizationStep`](api.md/#l2co.OptimizationStep) class represents the optimization algorithm used to minimize the objective function defined in a `Task` in a human-friendly way. It is used to abstract the different third-party packages that are used to implement the optimizers. It encapsulates the following elements:

- **optimizer** (`optimizer`): A string argument specifying the optimization algorithm to be used. The string argument is mapped to a registry of known optimizers ([`optimizer_mapping`](api.md/#l2co.optimizer_mapping)).
- **stopping_criteria** (`stopping_criteria`): A string argument specifying the stopping criteria for the optimization process. The string argument is mapped to a registry of known stopping criteria.
- **hyperparameters** (`hyperparameters`): A dictionary containing hyperparameters specific to the chosen optimizer. The hyperparameters are passed to the instantiation of the optimizer.

Currently, `l2co` supports the following JAX-based libraries: 

- [`optax`](https://github.com/google-deepmind/optax) for gradient-based optimization
- [`evosax`](https://github.com/RobertTLange/evosax) for gradient-free optimization. 

Although all the available optimizers from these libraries can be used, `l2co` provides a registry of commonly used optimizers that can be referenced by their string names.

In addition to the algorithms shipped by these libraries, `l2co` provides a built-in implementation of `SHADE` (success-history based adaptive differential evolution, Tanabe & Fukunaga 2013), registered as `'shade'`. Its optional turning-based mutation (Sun et al. 2020) is enabled with the `use_turning` hyperparameter, which additionally requires finite box bounds and a `max_fevals` hyperparameter set to the run's total fitness-evaluation budget.

On stochastic tasks whose loss takes an explicit PRNG key (e.g. the BBOB-noisy suite, `task.pass_rng`), the `'lbfgs'` entry (`lbfgs_update` in `l2co-optimizers`) draws an independent noise realization for every zoom-linesearch trial evaluation, matching the one-realization-per-feval contract of the evosax optimizers (plain `optax.lbfgs` would freeze one realization per outer step). On deterministic tasks no key is ever passed to the loss and the entry behaves exactly as plain `optax.lbfgs`.

Based on the given information, the [`OptimizationStep`](api.md/#l2co.OptimizationStep) class generated a `tag` attribute that identifies the optimizer and its configuration similarly to the `Task` class.

In [ ]:
from l2co import OptimizationStep

adam_step = OptimizationStep(
    optimizer='adam',
    stopping_criteria={'type': 'gradient_based', 'grad_tol': 1e-8},
    hyperparameters={'learning_rate': 1e-3}
)

sepcmaes_step = OptimizationStep(
    optimizer='sepcmaes',
    stopping_criteria={'type': 'covariance_matrix', 'tol': 1e-12},
)

adam_step

The `stopping_criteria` parameter allows you to define when the optimization process should stop. This criteria is evaluated every iteration. Currently, the following stopping criteria are supported:

- **`'gradient_based'`**: Stops the optimization when the norm of the gradient falls below a specified threshold.
- **`'loss_based'`**: Stops the optimization when the change in loss value between iterations falls below a specified threshold.
- **`'combined'`**: Combines the `'gradient_based'` and `'loss_based'` criteria, stopping the optimization when either condition is met.
- **`'covariance_matrix'`**: Stops the optimization when the maximum eigenvalue of the covariance matrix of the population (for evolutionary strategies) falls below a specified threshold.

The [`OptimizationStep`](api.md/#l2co.OptimizationStep) object can be saved to a JSON file ([`OptimizationStep.save`](api.md/#l2co.OptimizationStep.save)) and loaded again ([`OptimizationStep.load`](api.md/#l2co.OptimizationStep.load)) for later use.

## Budget function

In order to solve an optimization problem using `l2co`, you need to define a maximum number of iterations to specify how many update steps should be performed for a given task, assuming that the stopping criteria is not met before reaching this limit. This is done using a budget function.

Currently there are 2 budget functions available in `l2co`:

- **`fixed_budget`**: A simple budget function that always returns a fixed integer value representing the maximum number of iterations.
- **`based_on_dimensionality`**: A budget function that calculates the maximum number of iterations based on the dimensionality of the task (`Task.dimensionality`). The budget is computed as `budget_per_dimension * dimensionality`. An optional `max_iterations` parameter can be provided to cap the budget at a fixed upper limit, which is useful when working with high-dimensional tasks where the linear scaling would produce an impractically large budget.

In [ ]:
from l2co.budget import based_on_dimensionality, fixed_budget

The budget function takes any arguments and returns an integer value representing the maximum number of iterations for the optimization process.

In [ ]:
from functools import partial

budget_fn = partial(based_on_dimensionality, budget_per_dimension=1000)

print(f"The maximum budget is: {budget_fn(rastrigin_function)} iterations!")

# With a cap on the maximum iterations
budget_fn_capped = partial(based_on_dimensionality, budget_per_dimension=1000, max_iterations=5000)
print(f"The capped budget is: {budget_fn_capped(rastrigin_function)} iterations!")

## Samplers

In order to initialize the parameters of the model to be optimized, `l2co` provides several samplers that can be used to generate initial values. These samplers are located in the `l2co.samplers` module and can be used to create different types of distributions for initializing the model parameters.

The following samplers are available in `l2co.samplers`:

- **[`random_sampling`](api.md/#l2co.sampling.random_sampling)**: Samples parameters uniformly at random within a specified range.
- **[`xavier_sampling`](api.md/#l2co.sampling.xavier_sampling)**: Samples parameters using Xavier initialization, which is commonly used for neural networks.
- **[`normal_sampling`](api.md/#l2co.sampling.normal_sampling)**: Samples parameters from a normal (Gaussian) distribution with specified mean and standard deviation.
- **[`grid_sampling`](api.md/#l2co.sampling.grid_sampling)**: Samples parameters on a grid within a specified range, useful for low-dimensional problems or hyperparameter sweeps.
- **[`constant_sampling`](api.md/#l2co.sampling.constant_sampling)**: Fills all parameters with a single constant value. Useful for creating baseline initial conditions.

In [ ]:
from l2co.sampling import random_sampling, xavier_sampling, normal_sampling, grid_sampling, constant_sampling
import jax.random as jr
import equinox as eqx

# Partition the model parameters into inexact arrays (the parameters to be sampled) and static data
params, static = eqx.partition(rastrigin_function.model, eqx.is_inexact_array)

normal_sampling(key=jr.key(0), params=params, n_samples=5, mean=0.0, std=0.1)

## Running optimization

Now that we have identified the components required to run an optimization using `l2co`, we can proceed to set up and execute the optimization process. This involves creating instances of the [`Task`], [`OptimizationStep`](api.md/#l2co.OptimizationStep) and defining a budget function and a sampler function. Once everything is set up, we can create an instance of the [`RolloutWrapper`](api.md/#l2co.RolloutWrapper) class to manage the optimization process:

In [ ]:
from l2co import RolloutWrapper

In [ ]:
rollout_wrapper = RolloutWrapper.init(
    optimizer=adam_step,
    task=rastrigin_function,
    sampler=normal_sampling,
    key=jr.key(42),
    bounded=(-5.0, 5.0), # Optional box constraints for the loss function space
    verbose=True,
)

The [`RolloutWrapper.init`](api.md/#l2co.RolloutWrapper.init) method initializes internally a [`RunState`](api.md/#l2co.RunState) object to keep track of the optimization progress and results. This object tracks the following information during the optimization:

- **[`history_state`](api.md/#l2co.HistoryState)**: An instance of the [`HistoryState`](api.md/#l2co.HistoryState) class that records the entire history of the optimization process, including the mean, minimum and standard deviation value of each population at each iteration. Note; it only records these statistics per iteration, so for population-based optimizers, it does not store the individual members of the population!
- **`opt_state`**: The internal state of the optimizer being used, which is updated at each iteration based on the optimization algorithm. This is eather an [`optax.GradientTransformation`](https://optax.readthedocs.io/en/latest/api/transformations.html#optax.GradientTransformation), [`optax.GradientTransformationExtraArgs`](https://optax.readthedocs.io/en/latest/api/transformations.html#optax.GradientTransformationExtraArgs) or an `evosax.EvoState`, depending on the optimizer being used.
- **[`batch_state`](api.md/#l2co.BatchState)**: Optional state information related to data batching, if applicable to the task being optimized. It only stores the indices of the samples being used in the current batch.
- **[`update_class`](api.md/#l2co.UpdateClass)**: An instance of the [`UpdateClass`](api.md/#l2co.UpdateClass) class that encapsulates the optimizer initialization, update step and stopping criteria being used for the optimization process.
- **`max_iterations`**: The maximum number of iterations allowed for the optimization process, as defined by the budget function.

In [ ]:
rollout_wrapper.run_state

The [`RolloutWrapper.batch_evaluate`](api.md/#l2co.RolloutWrapper.batch_evaluate) method is used to run the optimization process for a set number of different realizations. For each realization a different random key is used to initialize the model parameters and optimizers parameters.

In [ ]:
run_state, batch_state, history_state = rollout_wrapper.batch_evaluate(key=jr.key(123), n_iterations=budget_fn(rastrigin_function), n_realizations=50)

This function returns the [`HistoryState`](api.md/#l2co.HistoryState) object from the [`RunState`](api.md/#l2co.RunState) that contains the recorded history of the optimization process across all realizations.

This JAX Array object can be further analyzed by converting it to an xarray object with [`history_to_xarray_realizations`](api.md/#l2co.history_to_xarray_realizations). It accepts an optional `budget_steps` parameter for interpolating the history to a fixed number of budget steps, which is useful for normalizing trajectories of different lengths.

Additionally, [`history_to_dataloader`](api.md/#l2co.history_to_dataloader) can convert the history directly into a [`DataLoader`](api.md/#l2co.DataLoader) object for further analysis, given the corresponding `Task` and `OptimizationStep`.

In [ ]:
from l2co import history_to_xarray_realizations

xarr_history = history_to_xarray_realizations(history_state)
xarr_history

If you want to run multiple optimizers on different tasks, the [`RunOptimization`](api.md/#l2co.RunOptimization) class provides a higher-level interface to manage and execute multiple optimization runs in a structured way. It is inherited from a `f3dasm.DataGenerator` object and can be used in conjunction with `f3dasm`'s data handling capabilities.

The class accepts several additional configuration parameters: `clear_jax_cache` (whether to free JAX caches after each run, default `True`), `verbose` (enable logging output, default `False`), and `save_input_parameters` (whether to include input parameter histories in the output, default `False`).

In [ ]:
from l2co import RunOptimization

data_generator = RunOptimization(sampler=normal_sampling, budget_fn=budget_fn, seed=3,
                n_realizations=3, bounded=(-5.0, 5.0))

We can create a design-of-experiments with different `Task` and `OptimizationStep` input parameters.

In [ ]:
from f3dasm import ExperimentData
from f3dasm.design import Domain

domain = Domain()
domain.add_parameter(name='task', to_disk=True, store_function=Task.save, load_function=Task.load)
domain.add_parameter(name='optimizer schedule', to_disk=True, store_function=OptimizationStep.save, load_function=OptimizationStep.load)

input_data= [{'task': rastrigin_function, 'optimizer schedule': adam_step},
             {'task': rastrigin_function, 'optimizer schedule': sepcmaes_step},
             {'task': sphere_function, 'optimizer schedule': adam_step},
             {'task': sphere_function, 'optimizer schedule': sepcmaes_step},
             ]

experiment_data = ExperimentData(domain=domain, input_data=input_data)
experiment_data

Calling the data generator will execute all the optimization runs and return a NetCDF dataset containing the results per combination of task and optimizer:

In [ ]:
experiment_data = data_generator.call(data=experiment_data, mode='sequential')
experiment_data

The 'create_optimization_dataset_new' experiment allows you to easily set up and run a series of optimization experiments using different tasks and optimizers.

## Analyzing results

After running the optimization experiments, we can analyze the results. There are two approaches:

1. **Direct analysis from `HistoryState`**: Convert the history directly to a [`DataLoader`](api.md/#l2co.DataLoader) using [`history_to_dataloader`](api.md/#l2co.history_to_dataloader).
2. **Batch analysis with `create_combined_odata`**: When using `RunOptimization` with `ExperimentData`, combine all individual NetCDF result files into a consolidated `.odata` file using [`create_combined_odata`](api.md/#l2co.create_combined_odata), then load with [`DataLoader.from_file`](api.md/#l2co.DataLoader.from_file).

The [`DataLoader`](api.md/#l2co.DataLoader) holds both the loss trajectories and the ERT performance metrics, and provides methods for computing performance statistics, creating ERTD profiles, and filtering by task or optimizer.

### Converting HistoryState to a DataLoader

The simplest way to analyze results from a single run is to convert the `HistoryState` directly into a `DataLoader` using `history_to_dataloader`:

In [ ]:
from l2co import DataLoader, history_to_dataloader

# Convert history_state from the earlier batch_evaluate call to a DataLoader
data_loader = history_to_dataloader(
    history_state,
    task=rastrigin_function,
    optimizer=adam_step,
)

data_loader

### Plotting with ERTD profiles

The `DataLoader` can be used together with the `l2co.plotting` module to visualize optimization performance. The [`plot_ertd_profile`](api.md/#l2co.plotting.plot_ertd_profile) function shows the run-time distribution for the given data, using the loader's performance variables:

In [ ]:
import l2co.plotting as plotting

fig, ax = plotting.plot_ertd_profile(data_loader)

### Inspecting the xarray data

The underlying data is stored as an `xarray.Dataset`, which you can access directly for custom analyses:

In [ ]:
# Access the underlying xarray dataset
data_loader.data